In [1]:
import numpy as np
import pickle
import itertools
from itertools import combinations

def count_higher_weight_faults(z, x, max_faults=3):
    n = z.shape[1]
    def to_mask(bits):
        return sum(1 << i for i, b in enumerate(bits) if b)
    # X stabilizer masks
    x_masks = [
        to_mask(row > 0)
        for row in x
    ]

    # Precompute single-fault error masks and syndromes
    faults = []

    for i, row in enumerate(z):
        for t in range(1, int(np.max(row))):
            error = to_mask(row > t)

            syndrome = sum(
                ((error & xm).bit_count() % 2) << j
                for j, xm in enumerate(x_masks)
            )

            faults.append(((i, t), error, syndrome))

    counts = {}

    for w in range(1, max_faults + 1):
        count = 0

        for combo in combinations(faults, w):
            error = 0
            syndrome = 0

            for _, e, s in combo:
                error ^= e
                syndrome ^= s

            if syndrome == 0 and error != 0:
                count += 1

        counts[w] = count

    return counts

In [3]:
defaultxschedule = np.load('hx.npy')[0:7]
defaultzschedule = np.load('hz.npy')[0:7]
print(defaultxschedule)

[[1 0 1 1 0 0 0 0 0 1 1 0 0 0 0 1 0 0]
 [1 1 0 0 1 0 0 0 0 0 1 1 0 0 0 0 1 0]
 [0 1 1 0 0 1 0 0 0 1 0 1 0 0 0 0 0 1]
 [0 0 0 1 0 1 1 0 0 1 0 0 1 1 0 0 0 0]
 [0 0 0 1 1 0 0 1 0 0 1 0 0 1 1 0 0 0]
 [0 0 0 0 1 1 0 0 1 0 0 1 1 0 1 0 0 0]
 [1 0 0 0 0 0 1 0 1 0 0 0 1 0 0 1 1 0]]
